# Stage 8 Hexagonal And Polygonal Optical Geometry

This notebook separates four different ideas that old outputs grouped
under "hexagon": a focal-plane polygon target, a hollow polygonal
outline, a phase-only focal-plane approximation, and a numerically
propagated hollow-polygon candidate.

A hexagonal or polygonal focal-plane pattern is not automatically a
propagation-stable Bessel-like beam. Propagation stability must be
measured with z-dependent metrics such as accepted depth, symmetry
retention, outline fidelity, core suppression, and side-lobe
contamination.

Phase-only SLM compatibility is not assumed for complex-amplitude
polygonal targets. If complex amplitude is required, the case is
labelled future_hardware_required or simulation_only unless a tested
encoding route is provided.


## Stage 8.7 Adjustable Quick-Look Guidance

<!-- STAGE87: adjustable quicklook guidance -->

For fast parameter scouting, use `notebooks/quicklook/00_quick_beam_to_sample_simulator.ipynb`. This notebook remains on its locked stage path: existing execution logic, propagation-power labels, material-proxy caveats, and governance routing are unchanged.

Safe local edits are the explicit config variables already exposed by this notebook, or a copied exploratory run. Keep `fail` and `marginal` labels visible. If a displayed image is visually smoothed, treat that as display interpolation only; rerun balanced/publication sampling before numerical interpretation.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import bessel_twin_core as bt
from vbb_study import setup_study
from vbb_study.equations import polygonal
from vbb_study.publication import advanced as advanced_schema
from vbb_study.studies import polygonal_cases

PATHS = setup_study.bootstrap(Path.cwd())
RUN_ID = PATHS.get("run_id") or None
CSV_OUT = PATHS["csv"] / "advanced"
CSV_OUT.mkdir(parents=True, exist_ok=True)
for folder in ("stage_h2", "polygonal_hex_ring", "hex_outline"):
    (PATHS["csv"] / folder).mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 120)


## Editable Notebook Controls

<!-- STAGE88: editable controls -->

This cell exposes the intended user-editable controls for exploratory runs. The locked stage logic below is preserved: changing these controls is for local investigation unless the notebook explicitly wires a value into a regenerated canonical output. Keep QA, caveats, and fail/marginal labels visible. For fast beam-to-sample exploration use the quicklook notebook; for publication-grade outputs use the locked stage runner.


In [ ]:
# STAGE88: visible editable controls for exploratory notebook use.
# Edit NOTEBOOK_CONTROLS below and re-run this cell to apply parameter
# overrides for downstream hexagonal/polygonal beam cells.
from vbb_study.publication import notebook_controls as nb_controls

NOTEBOOK_CONTROLS = nb_controls.make_notebook_controls(
    stage='advanced',
    # ── edit these for advanced geometry ────────────────────────────────────
    symmetry_order=6,
    propagation_tested=True,
)

# Wire control parameters to named variables used by downstream cells.
_p = NOTEBOOK_CONTROLS.parameters or {}
SYMMETRY_ORDER = int(_p.get("symmetry_order", 6))
PROPAGATION_TESTED = bool(_p.get("propagation_tested", True))

try:
    display(nb_controls.describe_controls(NOTEBOOK_CONTROLS))
except NameError:
    print(nb_controls.describe_controls(NOTEBOOK_CONTROLS).to_string(index=False))


In [ ]:
# Interactive beam quicklook — adjust sliders and click "Update plots".
# Runs a fast preview only; nothing is saved and this is independent of the
# hexagonal beam cells below.
from dataclasses import replace
from vbb_study.publication import notebook_widgets as nbw

_ql_base = bt.default_config("fast")
_panel = nbw.interactive_quicklook(_ql_base, method='holographic', preset='fast')
display(_panel)


## Geometry And Metric Helpers

The focal-plane target masks below are optical geometry targets. The
only row that is propagation-tested is explicitly propagated over z and
receives an accepted-depth metric. No row in this notebook claims
material writing, ablation, bonding, void formation, or a stable written
channel.


In [2]:
grid = bt.make_xy_grid(160, 0.30 * bt.um)
flat_radius_m = 7.0 * bt.um
line_width_m = 0.75 * bt.um
target_order = 6
z_values_m = np.linspace(0.0, 80.0 * bt.um, 9)

R = np.asarray(grid["R"], dtype=float)
PHI = np.asarray(grid["PHI"], dtype=float)
polygon_radius = polygonal.polygon_radius_function(PHI, flat_radius_m, target_order)
filled_mask = polygonal.polygonal_target_mask(
    R,
    PHI,
    flat_radius_m=flat_radius_m,
    N=target_order,
    hollow=False,
)
outline_mask = polygonal.polygonal_target_mask(
    R,
    PHI,
    flat_radius_m=flat_radius_m,
    N=target_order,
    line_width_m=line_width_m,
    hollow=True,
)
eval_mask = R <= (flat_radius_m / np.cos(np.pi / target_order) + 5.0 * bt.um)
target_outline_intensity = np.exp(-0.5 * ((R - polygon_radius) / line_width_m) ** 2)
target_outline_intensity = target_outline_intensity / (float(np.max(target_outline_intensity)) + bt.EPS)

def angular_order_metrics(intensity, grid, expected_order):
    I = np.asarray(intensity, dtype=float)
    Rg = np.asarray(grid["R"], dtype=float)
    PHIg = np.asarray(grid["PHI"], dtype=float) % (2.0 * np.pi)
    annulus = (Rg >= flat_radius_m - 2.0 * line_width_m) & (Rg <= polygonal.polygon_tip_radius_m(flat_radius_m, expected_order) + 2.0 * line_width_m)
    bins = np.linspace(0.0, 2.0 * np.pi, 361)
    profile = np.zeros(360, dtype=float)
    counts = np.zeros(360, dtype=float)
    idx = np.clip(np.digitize(PHIg[annulus], bins) - 1, 0, 359)
    np.add.at(profile, idx, I[annulus])
    np.add.at(counts, idx, 1.0)
    filled = counts > 0
    profile[filled] /= counts[filled]
    amps = np.abs(np.fft.rfft(profile - float(np.mean(profile))))
    if amps.size:
        amps[0] = 0.0
    measured = int(np.argmax(amps)) if amps.size else 0
    expected_amp = float(amps[int(expected_order)]) if int(expected_order) < amps.size else 0.0
    other = np.array(amps, copy=True)
    if int(expected_order) < other.size:
        other[int(expected_order)] = 0.0
    score = expected_amp / (expected_amp + float(np.max(other)) + bt.EPS)
    return measured, float(np.clip(score, 0.0, 1.0))

def focal_metrics(intensity, target_mask):
    I = np.asarray(intensity, dtype=float)
    norm = I / (float(np.max(I)) + bt.EPS)
    predicted = (norm >= 0.35) & eval_mask
    measured_order, symmetry = angular_order_metrics(norm, grid, target_order)
    return {
        "measured_symmetry_order": measured_order,
        "symmetry_score": symmetry,
        "outline_fidelity_score": polygonal.outline_fidelity_score(predicted, target_mask),
        "edge_uniformity_score": polygonal.edge_uniformity_score(norm, target_mask),
        "core_suppression_score": polygonal.core_suppression_score(
            norm,
            R,
            core_radius_m=0.45 * flat_radius_m,
            reference_mask=target_mask,
        ),
        "side_lobe_contamination_score": polygonal.side_lobe_contamination_score(
            norm,
            target_mask,
            evaluation_mask=eval_mask,
        ),
    }


## Focal-Plane Targets

These rows intentionally stop at the focal plane. They can be useful
target definitions or hardware-routing diagnostics, but they are not
propagation-stable beam claims.


In [3]:
rows = []
for case in polygonal_cases.polygonal_stage8_cases():
    if case["case_id"] == "hexagonal_focal_plane_target":
        intensity = filled_mask.astype(float)
        metrics = focal_metrics(intensity, filled_mask)
    elif case["case_id"] == "hollow_hexagonal_outline_target":
        intensity = target_outline_intensity
        metrics = focal_metrics(intensity, outline_mask)
    elif case["case_id"] == "phase_only_polygonal_approximation":
        phase_only_proxy = target_outline_intensity * (0.88 + 0.12 * np.cos(target_order * PHI) ** 2)
        intensity = phase_only_proxy / (float(np.max(phase_only_proxy)) + bt.EPS)
        metrics = focal_metrics(intensity, outline_mask)
    else:
        continue
    row = {
        **case,
        **metrics,
        "preset": "stage8_hexagonal_polygonal",
        "path": "focal_plane",
        "optical_model_status": "geometry_proxy",
        "material_model_status": "optical_only",
        "calibration_status": "uncalibrated",
        "accepted_depth_um": 0.0,
        "accepted_depth_definition": "not_applicable_focal_plane_only",
        "accepted_depth_fraction": 0.0,
        "canonical_zone_um": pd.NA,
        "strict_bessel_region_um": pd.NA,
    }
    rows.append(advanced_schema.annotate_advanced_beam_row(row, run_id=RUN_ID))
display(advanced_schema.ordered_advanced_beam_frame(rows)[[
    "case_id",
    "beam_family",
    "generation_method",
    "hardware_status",
    "propagation_stability_status",
    "outline_fidelity_score",
    "phase_only_compatible",
    "complex_amplitude_required",
]])


,case_id,beam_family,generation_method,hardware_status,propagation_stability_status,outline_fidelity_score,phase_only_compatible,complex_amplitude_required
0,hexagonal_focal_plane_target,hexagonal_polygonal,amplitude_phase_target,future_hardware_required,focal_plane_only,1.000000,False,True
1,hollow_hexagonal_outline_target,hollow_polygon,amplitude_phase_target,future_hardware_required,focal_plane_only,0.702509,False,True
2,phase_only_polygonal_approximation,hexagonal_polygonal,phase_only_slm,current_lab_realizable,focal_plane_only,0.717949,True,False


## Propagation-Tested Hollow Polygon Candidate

The field below is a simulation-only amplitude/phase target propagated
numerically. Its acceptance is based on z-dependent outline overlap,
sixfold retention, dark-core score, and side-lobe contamination. A
failure or marginal result remains useful because it prevents the
focal-plane target from being misread as a stable channel.


In [4]:
propagation_case = next(
    case for case in polygonal_cases.polygonal_stage8_cases()
    if case["case_id"] == "propagation_tested_hollow_polygon_candidate"
)
U0 = target_outline_intensity * np.exp(1j * 2.0 * PHI)
volume = bt.propagate_volume(
    U0,
    grid,
    1030.0 * bt.nm,
    z_values_m,
    n_medium=1.0,
    crop_pixels=160,
    bandlimit=True,
    method="bl_asm",
)
crop_grid = volume["crop_grid"]
Rc = np.asarray(crop_grid["R"], dtype=float)
PHIc = np.asarray(crop_grid["PHI"], dtype=float)
poly_c = polygonal.polygon_radius_function(PHIc, flat_radius_m, target_order)
outline_c = polygonal.polygonal_target_mask(
    Rc,
    PHIc,
    flat_radius_m=flat_radius_m,
    N=target_order,
    line_width_m=line_width_m,
    hollow=True,
)
eval_c = Rc <= (flat_radius_m / np.cos(np.pi / target_order) + 5.0 * bt.um)

z_rows = []
accepted = []
for idx, z_m in enumerate(z_values_m):
    plane = np.asarray(volume["intensity_stack"][idx], dtype=float)
    norm = plane / (float(np.max(plane)) + bt.EPS)
    predicted = (norm >= 0.35) & eval_c
    measured_order, symmetry = angular_order_metrics(norm, crop_grid, target_order)
    outline = polygonal.outline_fidelity_score(predicted, outline_c)
    edge = polygonal.edge_uniformity_score(norm, outline_c)
    core = polygonal.core_suppression_score(norm, Rc, core_radius_m=0.45 * flat_radius_m, reference_mask=outline_c)
    side = polygonal.side_lobe_contamination_score(norm, outline_c, evaluation_mask=eval_c)
    pass_plane = bool(outline >= 0.30 and symmetry >= 0.30 and core >= 0.50 and side <= 0.85)
    accepted.append(pass_plane)
    z_rows.append({
        **propagation_case,
        "case_id": f"{propagation_case['case_id']}_z{idx:02d}",
        "preset": "stage8_hexagonal_polygonal",
        "path": "simulation_z_profile",
        "z_um": float(z_m / bt.um),
        "accepted": pass_plane,
        "optical_model_status": "numerical_propagation",
        "material_model_status": "optical_only",
        "calibration_status": "uncalibrated",
        "measured_symmetry_order": measured_order,
        "symmetry_score": symmetry,
        "outline_fidelity_score": outline,
        "edge_uniformity_score": edge,
        "core_suppression_score": core,
        "side_lobe_contamination_score": side,
        "accepted_depth_definition": "z planes passing outline>=0.30, symmetry>=0.30, core>=0.50, side<=0.85",
    })

depth = polygonal.accepted_depth_from_metric_stack(z_values_m, accepted)
propagation_metrics = {
    "measured_symmetry_order": int(round(pd.Series([r["measured_symmetry_order"] for r in z_rows]).mode().iloc[0])),
    "symmetry_score": float(np.mean([r["symmetry_score"] for r in z_rows])),
    "outline_fidelity_score": float(np.mean([r["outline_fidelity_score"] for r in z_rows])),
    "edge_uniformity_score": float(np.mean([r["edge_uniformity_score"] for r in z_rows])),
    "core_suppression_score": float(np.mean([r["core_suppression_score"] for r in z_rows])),
    "side_lobe_contamination_score": float(np.mean([r["side_lobe_contamination_score"] for r in z_rows])),
    "accepted_depth_um": depth["accepted_depth_um"],
    "accepted_depth_fraction": depth["accepted_depth_fraction"],
    "accepted_plane_count": depth["accepted_plane_count"],
    "accepted_z_start_um": depth["accepted_z_start_um"],
    "accepted_z_end_um": depth["accepted_z_end_um"],
    "accepted_depth_definition": "longest contiguous z span passing outline/symmetry/core/side-lobe gate",
    "propagation_power_drift_fraction": float(
        (np.max(volume["total_power"]) - np.min(volume["total_power"]))
        / (np.mean(volume["total_power"]) + bt.EPS)
    ),
}
rows.append(advanced_schema.annotate_advanced_beam_row({
    **propagation_case,
    **propagation_metrics,
    "preset": "stage8_hexagonal_polygonal",
    "path": "simulation",
    "optical_model_status": "numerical_propagation",
    "material_model_status": "optical_only",
    "calibration_status": "uncalibrated",
}, run_id=RUN_ID))

z_profile = advanced_schema.ordered_advanced_beam_frame([
    advanced_schema.annotate_advanced_beam_row({
        **r,
        "accepted_depth_um": depth["accepted_depth_um"],
        "accepted_depth_fraction": depth["accepted_depth_fraction"],
        "propagation_power_drift_fraction": propagation_metrics["propagation_power_drift_fraction"],
    }, run_id=RUN_ID)
    for r in z_rows
])
display(z_profile[[
    "case_id",
    "z_um",
    "accepted",
    "outline_fidelity_score",
    "symmetry_score",
    "core_suppression_score",
    "side_lobe_contamination_score",
    "propagation_stability_status",
]])


,case_id,z_um,accepted,outline_fidelity_score,symmetry_score,core_suppression_score,side_lobe_contamination_score,propagation_stability_status
0,propagation_tested_hollow_polygon_candidate_z00,0.0,False,0.984848,0.253833,1.000000,0.351409,propagation_tested_fail
1,propagation_tested_hollow_polygon_candidate_z01,10.0,False,0.380583,0.420634,0.847911,1.063263,propagation_tested_fail
2,propagation_tested_hollow_polygon_candidate_z02,20.0,False,0.229117,0.563908,0.000000,1.835924,propagation_tested_fail
3,propagation_tested_hollow_polygon_candidate_z03,30.0,False,0.092179,0.725126,0.000000,1.830531,propagation_tested_fail
4,propagation_tested_hollow_polygon_candidate_z04,40.0,False,0.043873,0.732098,0.000000,2.566593,propagation_tested_fail
5,propagation_tested_hollow_polygon_candidate_z05,50.0,False,0.001359,0.533998,0.000000,2.847340,propagation_tested_fail
6,propagation_tested_hollow_polygon_candidate_z06,60.0,False,0.000000,0.317300,0.000000,3.669727,propagation_tested_fail
7,propagation_tested_hollow_polygon_candidate_z07,70.0,False,0.041582,0.516596,0.131851,1.791296,propagation_tested_fail
8,propagation_tested_hollow_polygon_candidate_z08,80.0,False,0.251762,0.585890,0.552882,1.137594,propagation_tested_fail


## Canonical Outputs

The canonical Stage 8 CSVs are written under `outputs/csv/advanced`.
Old `stage_h2`, `polygonal_hex_ring`, and `hex_outline` filenames are
refreshed as compatibility copies with the same native metadata.


In [5]:
summary = advanced_schema.ordered_advanced_beam_frame(rows)
acceptance = summary.copy()
acceptance["acceptance_check"] = acceptance["advanced_acceptance_label"]
acceptance["acceptance_pass"] = (
    acceptance["propagation_stability_status"].isin([
        "propagation_tested_pass",
        "propagation_tested_marginal",
    ])
    & ~acceptance["focal_plane_only"].astype(bool)
)

summary_path = CSV_OUT / "hexagonal_polygonal_beam_summary.csv"
acceptance_path = CSV_OUT / "hexagonal_polygonal_acceptance_summary.csv"
summary.to_csv(summary_path, index=False)
acceptance.to_csv(acceptance_path, index=False)

compatibility_summary = [
    PATHS["csv"] / "stage_h2" / "H2_air_knob_sweep.csv",
    PATHS["csv"] / "stage_h2" / "H2_survival_summary.csv",
    PATHS["csv"] / "stage_h2" / "H2_transient_hexlike_scan.csv",
    PATHS["csv"] / "polygonal_hex_ring" / "11_polygonal_hex_ring_acceptance_metrics.csv",
    PATHS["csv"] / "polygonal_hex_ring" / "11_polygonal_hex_ring_materials_proxy.csv",
    PATHS["csv"] / "polygonal_hex_ring" / "12_hollow_hex_sidelobe_ideal_sweep.csv",
    PATHS["csv"] / "polygonal_hex_ring" / "12_hollow_hex_sidelobe_lab_shortlist.csv",
    PATHS["csv"] / "hex_outline" / "13_hollow_hex_outline_checkpoint.csv",
    PATHS["csv"] / "hex_outline" / "14_hexlike_transient_vs_outline.csv",
    PATHS["csv"] / "hex_outline" / "15_hybrid_transient_seed_lab_gate.csv",
    PATHS["csv"] / "hex_outline" / "16_hex_bessel_like_summary.csv",
    PATHS["csv"] / "hex_outline" / "17_zernike_hex_bessel_sweep.csv",
]
for path in compatibility_summary:
    summary.to_csv(path, index=False)
z_profile.to_csv(PATHS["csv"] / "polygonal_hex_ring" / "11_polygonal_hex_ring_z_stability.csv", index=False)
z_profile.to_csv(PATHS["csv"] / "hex_outline" / "16_hex_bessel_like_z_profile.csv", index=False)

assert not summary["material_writing_success_claimed"].astype(bool).any()
assert not summary["stable_written_channel_claimed"].astype(bool).any()
assert not ((summary["focal_plane_only"].astype(bool)) & (summary["propagation_tested"].astype(bool))).any()
assert not (
    summary["complex_amplitude_required"].astype(bool)
    & summary["phase_only_compatible"].astype(bool)
).any()
display(summary[[
    "case_id",
    "beam_family",
    "model_level",
    "generation_method",
    "hardware_status",
    "propagation_stability_status",
    "advanced_acceptance_label",
    "material_writing_success_claimed",
    "stable_written_channel_claimed",
]])
print(summary_path)
print(acceptance_path)


,case_id,beam_family,model_level,generation_method,hardware_status,propagation_stability_status,advanced_acceptance_label,material_writing_success_claimed,stable_written_channel_claimed
0,hexagonal_focal_plane_target,hexagonal_polygonal,focal_plane_target,amplitude_phase_target,future_hardware_required,focal_plane_only,focal_plane_only,False,False
1,hollow_hexagonal_outline_target,hollow_polygon,focal_plane_target,amplitude_phase_target,future_hardware_required,focal_plane_only,focal_plane_only,False,False
2,phase_only_polygonal_approximation,hexagonal_polygonal,hardware_route,phase_only_slm,current_lab_realizable,focal_plane_only,focal_plane_only,False,False
3,propagation_tested_hollow_polygon_candidate,hollow_polygon,numerical_propagation,amplitude_phase_target,future_hardware_required,propagation_tested_fail,propagation_tested_fail,False,False


C:\PhD\Code\Publication_Study\outputs\csv\advanced\hexagonal_polygonal_beam_summary.csv
C:\PhD\Code\Publication_Study\outputs\csv\advanced\hexagonal_polygonal_acceptance_summary.csv
